In [6]:
import os
import json
import pandas as pd

### - 경로 설정

In [9]:
RAW_DIR = os.path.join("..", "data", "raw")
PROCEED_DIR = os.path.join("..", "data", "proceed")
INPUT_FILE = os.path.join(PROCEED_DIR, "kf_area_total_merged.csv")
FINAL_RAG_OUTPUT = os.path.join(PROCEED_DIR, "kf_area_rag_data.json")

print(f"입력 파일 경로: {INPUT_FILE}")
print(f"출력 파일 경로: {FINAL_RAG_OUTPUT}")

입력 파일 경로: ../data/proceed/kf_area_total_merged.csv
출력 파일 경로: ../data/proceed/kf_area_rag_data.json


### - 파일 로드하기

In [10]:
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
    print(f"데이터 로드 완료! (총 행 수: {len(df)}행)\n")
    
    # 결측치 진단 수행
    print("[데이터 품질 진단] 전체 컬럼별 결측치 현황:")
    print("-" * 50)
    null_counts = df.isnull().sum()
    null_percentages = (df.isnull().sum() / len(df)) * 100

    # 전체 컬럼의 결측치 현황 개수와 비율 출력
    for col in df.columns:
        print(f" - {col:<20} : {null_counts[col]:>6}개 누락 ({null_percentages[col]:>6.2f}%)")

    print("-" * 50)

    # RAG 역할별 결측치 집중 분석
    text_cols = ["sumry_cn", "data_title_nm", "relate_stry_nm", "addr"]
    meta_cols = ["data_manage_keyword", "ctprvn_nm", "signgu_nm", "theme_nm", "cl_nm", "sbjt_nm", "core_kwrd_cn", "relate_prsn_nm"]

    print("\n[RAG 비정형화 본문 컬럼] 결측치 현황:")
    for col in text_cols:
        if col in df.columns:
            print(f" ➔ {col:<20} : {null_counts[col]:>6}개 누락")

    print("\n[RAG 메타데이터 필터 컬럼] 결측치 현황:")
    for col in meta_cols:
        if col in df.columns:
            print(f" ➔ {col:<20} : {null_counts[col]:>6}개 누락")

    print("-" * 50)

    # 결측치가 하나라도 있는 행을 통째로 drop 했을 때의 시뮬레이션
    total_rows = len(df)
    df_dropped_all = df.dropna()  # 모든 컬럼 기준 결측치 제거
    left_rows_all = len(df_dropped_all)

    print("\n[위험 시뮬레이션] 데이터 전처리 방향성 결정 가이드:")
    print(f" ➔ 원본 전체 데이터 행 수: {total_rows}행")
    print(f" ➔ [케이스 A] '모든 컬럼' 중 결측치가 1개라도 있는 행을 무조건 삭제 시:")
    print(f"    - 남는 데이터: {left_rows_all}행 (유실율: {((total_rows - left_rows_all)/total_rows)*100:.2f}%)")

    # ➔ [케이스 B] "RAG 핵심 본문 컬럼(텍스트용)"만 결측치가 있을 때 삭제 시
    df_dropped_essential = df.dropna(subset=["sumry_cn", "data_title_nm"])
    left_rows_essential = len(df_dropped_essential)
    print(f" ➔ [케이스 B] '핵심 본문(제목, 요약내용)'에 결측치가 있는 행만 타겟 삭제 시:")
    print(f"    - 남는 데이터: {left_rows_essential}행 (유실율: {((total_rows - left_rows_essential)/total_rows)*100:.2f}%)")

else:
    print(f"[Error] 통합 마스터 파일({INPUT_FILE})이 존재하지 않습니다. 경로를 다시 확인해주세요.")

데이터 로드 완료! (총 행 수: 26876행)

[데이터 품질 진단] 전체 컬럼별 결측치 현황:
--------------------------------------------------
 - data_manage_no       :      0개 누락 (  0.00%)
 - data_title_nm        :      0개 누락 (  0.00%)
 - theme_nm             :      0개 누락 (  0.00%)
 - lwprt_theme_nm       :      0개 누락 (  0.00%)
 - cl_nm                :      0개 누락 (  0.00%)
 - lwprt_cl_nm          :      0개 누락 (  0.00%)
 - sbjt_nm              :      0개 누락 (  0.00%)
 - middl_sbjt_nm        :  15551개 누락 ( 57.86%)
 - sumry_cn             :      0개 누락 (  0.00%)
 - main_thumb_url       :  12297개 누락 ( 45.75%)
 - cntnts_url           :      0개 누락 (  0.00%)
 - ctprvn_nm            :   2442개 누락 (  9.09%)
 - signgu_nm            :   2575개 누락 (  9.58%)
 - addr                 :   3040개 누락 ( 11.31%)
 - ctlstt_la            :   3040개 누락 ( 11.31%)
 - ctlstt_lo            :   3040개 누락 ( 11.31%)
 - regist_de            :      0개 누락 (  0.00%)
 - relate_prsn_nm       :      0개 누락 (  0.00%)
 - relate_stry_nm       :   5135개 누락 ( 19.11%)
 

### - 시도명(ctprvn_nm) 결측치 제거 데이터 확인

In [11]:
# 
df_clean_sido = df.dropna(subset=["ctprvn_nm"]).reset_index(drop=True)

# 제거된 수치 계산
total_count = len(df)
removed_count = total_count - len(df_clean_sido)
drop_percentage = (removed_count / total_count) * 100

# 결과 리포트 출력
print(f"[시도명 필터링 결과 요약]")
print(f" ➔ 원본 마스터 데이터 행 수   : {total_count:>6}행")
print(f" ➔ 제거된 시도명 누락 데이터 수: {removed_count:>6}행 누락 (전체의 {drop_percentage:.2f}%)")
print(f" ➔ 시뮬레이터 가용 최종 데이터 수: {len(df_clean_sido):>6}행 (확보율 {100 - drop_percentage:.2f}%)")
print("-" * 60)

# 필터링 후 도메인(keyword)별 균형 데이터 확인
if "data_manage_keyword" in df_clean_sido.columns:
    cltur_count = len(df_clean_sido[df_clean_sido["data_manage_keyword"] == "cltur"])
    prsn_count = len(df_clean_sido[df_clean_sido["data_manage_keyword"] == "prsn"])
    print(f"[도메인별 데이터 밸런스 점검]")
    print(f" - 지역 이야기 + 예술인 (cltur) : {cltur_count:>6}행")
    print(f" - 지역 이야기 + 지역인물 (prsn): {prsn_count:>6}행")
else:
    print(""data_manage_keyword" 컬럼을 찾을 수 없어 도메인별 분류 확인을 건너뜁니다.")

[시도명 필터링 결과 요약]
 ➔ 원본 마스터 데이터 행 수   :  26876행
 ➔ 제거된 시도명 누락 데이터 수:   2442행 누락 (전체의 9.09%)
 ➔ 시뮬레이터 가용 최종 데이터 수:  24434행 (확보율 90.91%)
------------------------------------------------------------
[도메인별 데이터 밸런스 점검]
 - 지역 이야기 + 예술인 (cltur) :  13893행
 - 지역 이야기 + 지역인물 (prsn):  10541행


## [ RAG 전처리 파이프라인 전체 통합 코드 ]

In [13]:
# 1. 경로 설정
RAW_DIR = os.path.join("..", "data", "raw")
PROCEED_DIR = os.path.join("..", "data", "proceed")
INPUT_FILE = os.path.join(PROCEED_DIR, "kf_area_total_merged.csv")
FINAL_RAG_OUTPUT = os.path.join(PROCEED_DIR, "kf_area_rag_data.json")

print(f"입력 파일 경로: {INPUT_FILE}")
print(f"출력 파일 경로: {FINAL_RAG_OUTPUT}")
print("-" * 60)

# 2. 파일 로드하기
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
    print(f"데이터 로드 완료! (총 행 수: {len(df)}행)\n")
    
    # 3. 결측치 진단 수행
    print("[데이터 품질 진단] 전체 컬럼별 결측치 현황:")
    print("-" * 50)
    null_counts = df.isnull().sum()
    null_percentages = (df.isnull().sum() / len(df)) * 100

    for col in df.columns:
        print(f" - {col:<20} : {null_counts[col]:>6}개 누락 ({null_percentages[col]:>6.2f}%)")
    print("-" * 50)

    # RAG 역할별 결측치 집중 분석
    text_cols = ["sumry_cn", "data_title_nm", "relate_stry_nm", "addr"]
    meta_cols = ["data_manage_keyword", "ctprvn_nm", "signgu_nm", "theme_nm", "cl_nm", "sbjt_nm", "core_kwrd_cn", "relate_prsn_nm"]

    print("\n[RAG 비정형화 본문 컬럼] 결측치 현황:")
    for col in text_cols:
        if col in df.columns:
            print(f" ➔ {col:<20} : {null_counts[col]:>6}개 누락")

    print("\n[RAG 메타데이터 필터 컬럼] 결측치 현황:")
    for col in meta_cols:
        if col in df.columns:
            print(f" ➔ {col:<20} : {null_counts[col]:>6}개 누락")
    print("-" * 50)

    # 결측치 제거 시뮬레이션
    total_rows = len(df)
    df_dropped_all = df.dropna()
    left_rows_all = len(df_dropped_all)

    print("\n[위험 시뮬레이션] 데이터 전처리 방향성 결정 가이드:")
    print(f" ➔ 원본 전체 데이터 행 수: {total_rows}행")
    print(f" ➔ [케이스 A] '모든 컬럼' 중 결측치가 1개라도 있는 행을 무조건 삭제 시:")
    print(f"    - 남는 데이터: {left_rows_all}행 (유실율: {((total_rows - left_rows_all)/total_rows)*100:.2f}%)")

    df_dropped_essential = df.dropna(subset=["sumry_cn", "data_title_nm"])
    left_rows_essential = len(df_dropped_essential)
    print(f" ➔ [케이스 B] '핵심 본문(제목, 요약내용)'에 결측치가 있는 행만 타겟 삭제 시:")
    print(f"    - 남는 데이터: {left_rows_essential}행 (유실율: {((total_rows - left_rows_essential)/total_rows)*100:.2f}%)")
    print("-" * 60)

    # 4. 시도명(ctprvn_nm) 누락 데이터 필터링 (지도 UI 매칭용)
    df_clean_sido = df.dropna(subset=["ctprvn_nm"]).reset_index(drop=True)

    total_count = len(df)
    removed_count = total_count - len(df_clean_sido)
    drop_percentage = (removed_count / total_count) * 100

    print(f"[시도명 필터링 결과 요약]")
    print(f" ➔ 원본 마스터 데이터 행 수   : {total_count:>6}행")
    print(f" ➔ 제거된 시도명 누락 데이터 수: {removed_count:>6}행 누락 (전체의 {drop_percentage:.2f}%)")
    print(f" ➔ 시뮬레이터 가용 최종 데이터 수: {len(df_clean_sido):>6}행 (확보율 {100 - drop_percentage:.2f}%)")
    print("-" * 60)

    # 도메인(keyword)별 균형 데이터 확인
    if "data_manage_keyword" in df_clean_sido.columns:
        cltur_count = len(df_clean_sido[df_clean_sido["data_manage_keyword"] == "cltur"])
        prsn_count = len(df_clean_sido[df_clean_sido["data_manage_keyword"] == "prsn"])
        print(f"[도메인별 데이터 밸런스 점검]")
        print(f" - 지역 이야기 + 예술인 (cltur) : {cltur_count:>6}행")
        print(f" - 지역 이야기 + 지역인물 (prsn): {prsn_count:>6}행")
    print("-" * 60)


    # 5. RAG 데이터셋 위한 자연어 합성 함수 정의
    def synthesize_simulator_context(row):
        if row["data_manage_keyword"] == "cltur":
            domain_ctx = "지역 이야기와 예술인에 관한 역사 기록"
            identity_tag = "예술인"
        else:
            domain_ctx = "지역 이야기와 지역 인물에 관한 역사 기록"
            identity_tag = "지역 인물"
            
        title = row["data_title_nm"]
        summary = row["sumry_cn"]
        related_person = row["relate_prsn_nm"]
        
        # 필수 문장 빌드 (시도명 보장됨)
        text_chunks = [
            f"[시뮬레이터 단서] 이 문서는 '{row["ctprvn_nm"]}' 지역의 {domain_ctx}입니다.",
            f"자료의 명칭은 '{title}'이며, 관련된 역사적 사건 및 핵심 요약은 다음과 같습니다: '{summary}'",
            f"이 사건 및 현장과 밀접하게 연관된 핵심 {identity_tag}은 '{related_person}'입니다."
        ]
        
        # 선택 데이터 방어 코드 (존재할 때만 합류)
        if pd.notna(row.get("addr")) and str(row.get("addr")).strip() != "":
            text_chunks.append(f"이 역사적 현장의 구체적인 위치 및 주소는 '{row["addr"]}' 입니다.")
            
        if pd.notna(row.get("core_kwrd_cn")) and str(row.get("core_kwrd_cn")).strip() != "":
            text_chunks.append(f"이 단서와 매칭되는 주요 역사 키워드는 [{row["core_kwrd_cn"]}] 입니다.")
            
        if pd.notna(row.get("relate_stry_nm")) and str(row.get("relate_stry_nm")).strip() != "":
            text_chunks.append(f"이 장소와 인물에 얽힌 구체적인 비화 및 지역 이야기는 다음과 같습니다: '{row["relate_stry_nm"]}'")
            
        return " ".join(text_chunks)


    # 6. 최종 추출 및 포맷팅 루프 가동
    print("최종 RAG 스키마 매핑 및 텍스트 합성 중...")
    rag_dataset = []

    for _, row in df_clean_sido.iterrows():
        # NaN 결측치를 JSON 호환용 None으로 매핑
        row_dict = row.where(pd.notna(row), None).to_dict()
        
        # 자연어 줄글 본문 생성
        combined_context = synthesize_simulator_context(row)
        
        # 시군구명 미세 누락 방어
        sigungu_filter = row_dict.get("signgu_nm") if row_dict.get("signgu_nm") else "미지정"
        
        # 메타데이터 구조화
        metadata = {
            "source_id": row_dict.get("data_manage_no"),
            "domain_type": row_dict.get("data_manage_keyword"), # cltur 또는 prsn
            "theme": row_dict.get("theme_nm"),
            "category": row_dict.get("cl_nm"),
            "subject": row_dict.get("sbjt_nm"),
            "region_sido": row_dict.get("ctprvn_nm"),
            "region_sigungu": sigungu_filter,
            "related_person": row_dict.get("relate_prsn_nm"),
            "latitude": row_dict.get("ctlstt_la"),
            "longitude": row_dict.get("ctlstt_lo")
        }
        
        rag_dataset.append({
            "text": combined_context,
            "metadata": metadata
        })

    # 7. JSON 저장 (개행 형태가 아닌 단일 정돈 포맷)
    with open(FINAL_RAG_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(rag_dataset, f, ensure_ascii=False, indent=2)
        
    print("-" * 60)
    print(f"[RAG 데이터셋 최종 빌드 및 저장 완료!]")
    print(f" ➔ 마스터 파일 위치: {FINAL_RAG_OUTPUT}")
    print(f" ➔ 저장된 가용 데이터: {len(rag_dataset)}개 행 완료")

else:
    print(f"[Error] {INPUT_FILE} 파일이 존재하지 않습니다. 경로를 다시 확인해주세요.")

입력 파일 경로: ../data/proceed/kf_area_total_merged.csv
출력 파일 경로: ../data/proceed/kf_area_rag_data.json
------------------------------------------------------------
데이터 로드 완료! (총 행 수: 26876행)

[데이터 품질 진단] 전체 컬럼별 결측치 현황:
--------------------------------------------------
 - data_manage_no       :      0개 누락 (  0.00%)
 - data_title_nm        :      0개 누락 (  0.00%)
 - theme_nm             :      0개 누락 (  0.00%)
 - lwprt_theme_nm       :      0개 누락 (  0.00%)
 - cl_nm                :      0개 누락 (  0.00%)
 - lwprt_cl_nm          :      0개 누락 (  0.00%)
 - sbjt_nm              :      0개 누락 (  0.00%)
 - middl_sbjt_nm        :  15551개 누락 ( 57.86%)
 - sumry_cn             :      0개 누락 (  0.00%)
 - main_thumb_url       :  12297개 누락 ( 45.75%)
 - cntnts_url           :      0개 누락 (  0.00%)
 - ctprvn_nm            :   2442개 누락 (  9.09%)
 - signgu_nm            :   2575개 누락 (  9.58%)
 - addr                 :   3040개 누락 ( 11.31%)
 - ctlstt_la            :   3040개 누락 ( 11.31%)
 - ctlstt_lo            :   3